# 실습: Groq 제어 파라미터

같은 프롬프트라도 파라미터를 바꾸면 길이, 다양성, 중단 위치, 모델별 품질이 달라집니다.


---
## 실습 전 주의

- 모든 모델이 모든 파라미터 조합에서 같은 품질을 내지는 않습니다.
- Groq는 빠른 응답 확인에 강점이 있으므로 같은 질문을 여러 모델로 비교하기 좋습니다.
- 비교 실습은 비용을 줄이기 위해 `max_completion_tokens`를 작게 둡니다.
---


In [1]:
import os

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 GROQ_API_KEY를 먼저 설정하세요.")

MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
client = Groq(api_key=api_key)

print(f"사용 모델: {MODEL}")


사용 모델: llama-3.1-8b-instant


In [2]:
def show_usage(response) -> None:
    usage = getattr(response, "usage", None)
    if usage:
        print(
            f"\n토큰: prompt={usage.prompt_tokens}, "
            f"completion={usage.completion_tokens}, total={usage.total_tokens}"
        )


def generate(label: str, prompt: str, model: str | None = None, system: str | None = None, **params):
    print(f"\n--- {label} ---")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    try:
        response = client.chat.completions.create(
            model=model or MODEL,
            messages=messages,
            **params,
        )
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        print(f"Groq API 오류: {status_code} {type(exc).__name__}")
        print(exc)
        return None

    print(response.choices[0].message.content)
    show_usage(response)
    return response


---
## 1. 기준 프롬프트


In [3]:
base_prompt = (
    "초보 개발자가 함수형 프로그래밍을 이해할 수 있도록 "
    "핵심 개념과 Python 예시를 설명해줘."
)

baseline = generate(
    "기준 응답",
    base_prompt,
    max_completion_tokens=260,
)



--- 기준 응답 ---
함수형 프로그래밍은 프로그램을 함수로 구성하는 프로그래밍 패러다임입니다. 함수형 프로그래밍에서는 데이터를 변환하고 처리하는 데 사용되는 함수를 가조차 수정하지 않는 방식으로 프로그램을 작성하는 것을 목표로 합니다. 이것은 프로그램의 의존성과 오류를 줄이기 위한 것입니다.

**함수형 프로그래밍의 핵심 개념**

1. **순수 함수**: 순수 함수는 동일한 입력에 대해 항상 동일한 출력을 반환하는 함수입니다. 순수 함수에는 부정확성, 시간적 결합성, 부차적 효과가 없습니다.
2. **클로저**: 클로저는 함수 내부에서 외부 변수를 캡처하는 함수를 말합니다. 클로저는 외부 변수의 값을 유지하고, 외부 변수를 참조하므로, 외부 변수의 값이 변경될 때 클로저의 값을 변경할 수 있습니다.
3. **함수 합성**: 함수 합성은 두 개 이상의 함수를 연쇄적으로 호출하여 결과를 반환합니다. 함수 합성은 함수 간에 데이터를 전달하고, 함수의 결과를 다른 함수에 입력으로 사용할 수

토큰: prompt=64, completion=260, total=324


---
## 2. `max_completion_tokens`: 출력 길이 제한


In [4]:
for limit in [60, 160, 320]:
    generate(
        f"max_completion_tokens={limit}",
        base_prompt,
        max_completion_tokens=limit,
    )



--- max_completion_tokens=60 ---
함수형 프로그래밍은 객체 지향 프로그래밍과 함께 일반적인 프로그래밍 방법 중 하나입니다. 함수형 프로그래밍은 함수 자체를 값으로 다루는 프로그래밍 기법을 통해 코드를 작고 독립적이고

토큰: prompt=64, completion=60, total=124

--- max_completion_tokens=160 ---
함수형 프로그래밍은 프로그래밍 패턴과 개념으로, 코드를 작성할 때 **함수**라는 개념에 집중하여 프로그램을 표현하기 위한 방법입니다. 함수형 프로그래밍은 **가정**이 없이, **기존 데이터**를 변경하지 않고, **기존 데이터**의 조합을 만드는 것을 목표로 합니다.

### 1. Immutable (불변성)

함수형 프로그래밍의 하나의 핵심 개념이 바로 **불변성**입니다. 불변성이란 데이터의 **원본**을 변경하는 것이 아닌, 새로운 데이터를 생성하여 작업을 완료하는 개념입니다. 예를 들어, 기존 데이터가 `[1, 2

토큰: prompt=64, completion=160, total=224

--- max_completion_tokens=320 ---
함수형 프로그래밍은 변수를 사용하는 반대되는 개념으로, 프로그램을 구성하는 변수의 변경이 최소화가 목표입니다. 이 개념은 함수를 사용하여 복잡한 프로그램을 구조화하는 데 도움이 됩니다.

### 1. 순수 함수(pure function)

순수 함수는 입력 값에서 출력 값을 구할 수 있는 함수입니다. 또한 순수 함수의 입력 값은 내부 상태에 영향을 받지 않고, 내부 상태도 함수를 호출할 때마다 고정되어야 합니다.

```python
def add(x, y):
    return x + y

# 순수 함수 인 add는 내부 상태에 의존하지 않습니다.
# add 함수의 반환 값은 항상 x와 y의 합입니다.
```

### 2. 중복 제거 (Higher Order Function)

함수형 프로그래밍에서는 변수를 사용하지 않고, 함수를 

---
## 3. `temperature`: 무작위성 조절

낮을수록 안정적이고 비슷한 답변이 나오기 쉽습니다.
높을수록 표현이 다양해질 수 있습니다.


In [5]:
creative_prompt = "데이터 분석가를 위한 Python 학습 슬로건을 3개 만들어줘."

for temperature in [0.1, 0.7, 1.2]:
    generate(
        f"temperature={temperature}",
        creative_prompt,
        temperature=temperature,
        max_completion_tokens=180,
    )



--- temperature=0.1 ---
데이터 분석가가 되기 위한 Python 학습 슬로건 3개는 다음과 같습니다.

1. **"Python을 마스터하고, 데이터를 분석하라!"** - 데이터 분석가가 되기 위해서는 Python 프로그래밍 언어를 잘 이해하고, 데이터를 분석하고 시각화하는 방법을 배워야 합니다.
2. **"데이터를 파악하고, 의미를 찾아라!"** - 데이터 분석가는 데이터를 파악하고, 의미를 찾아내는 것을 목표로 합니다. Python을 사용하여 데이터를 분석하고, 시각화하여 의미를 찾아내는 방법을 배워야 합니다.
3. **"데이터를 사용하여, 결정을 내라!"** - 데이터 분석가는 데이터를 사용하여 결정을 내리는 것을 목표로 합니다. Python을 사용하여 데이터를 분석하고, 시

토큰: prompt=54, completion=180, total=234

--- temperature=0.7 ---
데이터 분석가의 지식과 기술을 향상시키고, 데이터를 통찰력 있게 분석하고 가치 있는 결정을 내리기 위해 Python을 배우는 데 도움이 될 수 있는 슬로건 3개를 아래에 제시합니다.

1. **"코드는 데이터, 데이터는 코드로 시작된다."** - 데이터 분석에 사용되는 코드와 데이터는 서로 밀접한 관계를 맺고 있다. 따라서 Python으로 데이터를 분석하려면 코드를 작성하는 데 집중해야 한다.

2. **"데이터를 이해하려면, 코드를 이해해야한다."** - 데이터 분석은 코드를 통해 데이터를 이해하고 해석하는 과정이기도 하다. 코드를 이해하는 데 집중한다면 데이터 분석에 도움이 될 것이다.

3. **"코드를 작성하는 것은 데이터

토큰: prompt=54, completion=180, total=234

--- temperature=1.2 ---
데이터 분석가로서 Python을 공부하는 데 도움이 될 수 있는 슬로건을 세個 제시합니다.

1. **Python은 데이터와 함께 배우기, 데이터 분석을위한 열기**
 
  - 이 슬로기는 데이터 분석을

---
## 4. `top_p`: 후보 토큰 범위 조절

`top_p`는 누적 확률 기준으로 후보 토큰의 범위를 제한합니다.
보통 `temperature`와 동시에 크게 바꾸기보다 하나씩 비교합니다.


In [6]:
for top_p in [0.4, 0.8, 1.0]:
    generate(
        f"top_p={top_p}",
        creative_prompt,
        top_p=top_p,
        max_completion_tokens=180,
    )



--- top_p=0.4 ---
데이터 분석가의 성장과 성취를 위해 다음과 같은 3개의 슬로건을 제안합니다.

1. **"데이터를 이해하고, 지식을 쌓고, 분석을 향상하라."**
 - 데이터 분석가는 데이터를 분석하고, 이해하는 데 중점을 두어야 합니다. 또한 지식을 쌓고, 자신의 분석 기술을 향상시키는 노력을 기울여야 합니다.

2. **"데이터가 말해주지 않는 이야기를 찾아라."**
 - 데이터 분석가는 데이터에 있는 것을 넘어서, 데이터가 말해주지 않는 이야기를 찾아내야 합니다. 이는 데이터를 더 깊이 이해하고, 새로운 통찰력을 얻는 데 도움이 됩니다.

3. **"데이터가 가르쳐주는 교훈을 실천하라."**
 - 데이터 분석

토큰: prompt=54, completion=180, total=234

--- top_p=0.8 ---
데이터 분석가로 성장하기 위한 Python 학습 슬로건 3개입니다.

1. **코드는 데이터에 이야기하라**: Python을 사용하여 데이터를 청취하고, 데이터의 의미를 밝히는 코드를 작성하는 습관을 길러라.
2. **탐구적이며 의존하지 말라**: 데이터에 대한 궁금증을 해결하기 위해 Python을 사용하는 것과 마찬가지로, 의존하지 말라. 코드를 간결하고 직관적으로 작성하는 습관을 길러라.
3. **실습을 통한 성장**: Python을 사용하여 데이터를 분석하고, 시각화하는 실습을 통해 데이터 분석가로서의 역량을 개선하는 습관을 길러라.

토큰: prompt=54, completion=167, total=221

--- top_p=1.0 ---
데이터 분석가를 위한 Python 학습 슬로건 3개는 다음과 같습니다.

1. **파이썬은 데이터 과학의 언어**: 데이터 분석을 위한 기초적인 통계적/기계 학습 지식을 학습하고, 파이썬을 통해 데이터를 추출, 전처리, 분석 및 시각화 해 보세요.

2. **청산하기, 파악하기, 결과내기**: 데이터 세트를 청산하고 특징을 파악한 후, 파이썬을 통해 결과를 도출하여 이끌어 가세요.


---
## 5. `stop`: 특정 문자열에서 출력 중단


In [7]:
stop_prompt = (
    "다음 형식으로 세 단계를 작성해줘. "
    "각 단계 끝에는 ---끝---을 붙여줘. "
    "주제: API 키를 안전하게 관리하는 법"
)

generate(
    "stop 없이 실행",
    stop_prompt,
    max_completion_tokens=260,
)

generate(
    "stop=['---끝---'] 적용",
    stop_prompt,
    stop=["---끝---"],
    max_completion_tokens=260,
)



--- stop 없이 실행 ---
API 키를 안전하게 관리하는 법

### 단계 1: API 키 생성 및 저장

API 키를 생성하고 안전하게 저장하는 것이 중요합니다. 일반적으로 API 키는 API 제공자에서 제공하거나 직접 생성할 수 있습니다.

- **API 키 생성**: API 키를 생성하는 방법은 API 제공자마다 다릅니다. 예를 들어, Google Maps API의 경우 API 키를 Google Cloud 콘솔에서 생성할 수 있습니다.
- **API 키 저장**: 생성된 API 키를 안전하게 저장하는 것이 중요합니다. 다음과 같은 방법을 사용할 수 있습니다.
 + 환경 변수를 사용하여 API 키를 저장
 + 암호화된 텍스트 文件을 사용하여 API 키를 저장
 + 비밀 저장소(예: HashiCorp Vault)를 사용하여 API 키를 저장

### 단계 2: API 키 접근 제한

API 키를 안전하게 관리하기 위해 접근을 제한하는 것이 중요합니다. 다음과 같은 방법을 사용할 수 있습니다.

- **IP 주소 기반 접근 제한**: 특정 IP 주소에서만 API 키를 허용할 수 있습니다.
- **인증 기반 접근 제한**: API 키를 사용한 접근을 인증하는 시스템을 구

토큰: prompt=75, completion=260, total=335

--- stop=['---끝---'] 적용 ---
API 키를 안전하게 관리하는 법의 세 단계는 다음과 같다.

### 단계 1: API 키를 분리하고 암호화하기

API 키를 안전하게 관리하는 가장 첫 단계는 API 키를 분리하여 암호화하는 것이다. 
API 키를 암호화하면 외부에 노출되는 위험을 줄일 수 있다. 
이때, 환경 변수나 암호화된 파일에 API 키를 저장하는 것이 좋다. 
예를 들어, Python에서 `os.environ`을 사용하여 환경 변수로 API 키를 저장하거나, 
`cryptography`를 사용하여 API 키를 암호화할 수 있다. 

```
# 암호화된 파일을 만들기 위해 cryptog

ChatCompletion(id='chatcmpl-0b44f60d-8e97-4231-b9e4-ab238af1f18c', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='API 키를 안전하게 관리하는 법의 세 단계는 다음과 같다.\n\n### 단계 1: API 키를 분리하고 암호화하기\n\nAPI 키를 안전하게 관리하는 가장 첫 단계는 API 키를 분리하여 암호화하는 것이다. \nAPI 키를 암호화하면 외부에 노출되는 위험을 줄일 수 있다. \n이때, 환경 변수나 암호화된 파일에 API 키를 저장하는 것이 좋다. \n예를 들어, Python에서 `os.environ`을 사용하여 환경 변수로 API 키를 저장하거나, \n`cryptography`를 사용하여 API 키를 암호화할 수 있다. \n\n```\n# 암호화된 파일을 만들기 위해 cryptography를 사용\nfrom cryptography.fernet import Fernet\n\nAPI_KEY = "your_api_key"\ncipher_suite = Fernet.generate_key()\ncipher = Fernet(cipher_suite)\nencrypted_api_key = cipher.encrypt(API_KEY.encode())\n\n# 암호화된 API 키를 파일에 저장\nwith open(\'api_key.txt\', \'wb\') as f:\n    f.write(encrypted_api_key)\n```\n\n### 단계 2: API 키를 보호할 수단 구축하기\n\nAPI 키를 보호할 수단', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055397, model='llama-3

---
## 6. 시스템 메시지와 사용자 메시지 분리


In [8]:
prompt = "클래스와 객체의 차이를 알려줘."

generate(
    "일반 설명",
    prompt,
    max_completion_tokens=220,
)

generate(
    "면접 대비 설명",
    prompt,
    system=(
        "당신은 주니어 개발자 면접 코치입니다. "
        "정의, 비유, 면접 답변 예시 순서로 작성하세요."
    ),
    max_completion_tokens=340,
)



--- 일반 설명 ---
클래스와 객체는 객체 지향 프로그래밍(OOP)에서 중요한 개념입니다. 클래스와 객체의 차이를 설명드리겠습니다.

**클래스**

클래스는 객체의 설계도나 청사진입니다. 클래스는 특정 타입의 객체를 정의하는 데 사용하는 블록입니다. 클래스에는 데이터(속성)와 메서드(행위)가 정의됩니다.

클래스는 다음과 같은 내용을 포함할 수 있습니다.

- 속성(멤버 변수): 객체의 상태나 데이터를 설명하는 변수
- 메서드(멤버 함수): 객체의 행동이나 함수를 설명하는 함수

 예를 들어, 다음과 같이 클래스를 정의할 수 있습니다.

```c
class Car {
  private String brand; // 속성(멤버 변수)
  private int model;

  public Car(String brand, int model) {
    this.brand = brand;
    this.model = model;
  }

  public void startEngine() { // 메서드(멤버 함수)
    System

토큰: prompt=46, completion=220, total=266

--- 면접 대비 설명 ---
클래스와 객체는 객체지향 프로그래밍(OOP)에서 사용되는 핵심 개념입니다. 

**클래스**

클래스는 객체를 정의하고 설계하는 개념입니다. 클래스는 객체의 속성과 동작을 정의하는 블록으로, 객체의 구조와 행위를 표현할 수 있습니다. 클래스는 다음 세 가지 특성을 갖습니다.

- **속성**: 객체의 특성을 나타내는 변수나 필드입니다.
- **메서드**: 객체의 동작을 나타내는 함수 또는 메소드입니다.
- **상태**: 객체의 속성과 상태가 나타내는 정보입니다.

**객체**

객체는 클래스에 정의된 속성과 메서드를 가지는实제적인 존재입니다. 객체는 클래스에서 정의된 속성과 메서드를 가지고 있으며, 객체의 상태와 행동을 표현할 수 있습니다. 객체는 객체를 고유하게 구분하기 위해 고유한 식별자가 필요하며, 객체의 속성과 메

ChatCompletion(id='chatcmpl-a9f77bfb-9098-494a-98d9-b2c785a9ff46', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='클래스와 객체는 객체지향 프로그래밍(OOP)에서 사용되는 핵심 개념입니다. \n\n**클래스**\n\n클래스는 객체를 정의하고 설계하는 개념입니다. 클래스는 객체의 속성과 동작을 정의하는 블록으로, 객체의 구조와 행위를 표현할 수 있습니다. 클래스는 다음 세 가지 특성을 갖습니다.\n\n- **속성**: 객체의 특성을 나타내는 변수나 필드입니다.\n- **메서드**: 객체의 동작을 나타내는 함수 또는 메소드입니다.\n- **상태**: 객체의 속성과 상태가 나타내는 정보입니다.\n\n**객체**\n\n객체는 클래스에 정의된 속성과 메서드를 가지는实제적인 존재입니다. 객체는 클래스에서 정의된 속성과 메서드를 가지고 있으며, 객체의 상태와 행동을 표현할 수 있습니다. 객체는 객체를 고유하게 구분하기 위해 고유한 식별자가 필요하며, 객체의 속성과 메서드를 통해 객체를 구현합니다.\n\n클래스와 객체의 차이를 이해하기 위해 예를 떠올려 보겠습니다. \n\n예를 들어, "자동차" 라는 클래스를 만들었습니다. 이 클래스에는 "차 색상", "차 크기", "차 종류" 등의 속성과 "주차", "운전" 등이 가능한 메서드가 있습니다. 이 클래스를 기반으로 "소나타", "아반떼" 등의 자동차 객체를 만들면 됩니다.\n\n소나타 객체와 아반떼 객체는 고유한 색', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055399, model='llama-3.1-8b-instant', object='chat.comple

---
## 7. 모델 비교


In [9]:
compare_prompt = (
    "LLM API 수업에서 Groq를 OpenAI, Gemini와 함께 가르칠 때 "
    "Groq의 장점과 한계를 균형 있게 설명해줘."
)

models_to_compare = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
]

for model_name in models_to_compare:
    generate(
        f"모델 비교: {model_name}",
        compare_prompt,
        model=model_name,
        max_completion_tokens=320,
    )



--- 모델 비교: llama-3.1-8b-instant ---
Groq는 OpenAI, Gemini과 함께 LLM(넬러셔미디어 모델) API에 적용되는 플랫폼입니다. Groq를 설명하기에 앞서, Groq의 장점과 한계에 대해 살펴보겠습니다.

**Groq의 장점:**

1. **고성능**: Groq는 OpenAI, Gemini과 같은 고품질의 LLM을 실행할 수 있습니다. Groq의 하드웨어 인프라를 통해 LLM의 성능을 최악으로 제한하지 않으면서 높은 성능을 제공합니다.
2. **스칼라빌리티**: Groq는 높은 규모에서 확장할 수 있습니다. Groq의 플랫폼을 사용하면 LLM을 대량으로 생성하고 배포할 수 있습니다.
3. **편리함**: Groq는 LLM을 위한 소프트웨어 플랫폼을 제공합니다. 사용자는 LLM을 생성, 배포 및 관리하기 위해 Groq의 APIs를 사용할 수 있습니다.
4. **안정성**: Groq는 안정적인 플랫폼입니다. 많은 기업이 Groq을 사용하며 LLM을 운영하는데 안정적인 성능을 제공해주고 있습니다.

**Groq의 한계:**

1. **비용**: Groq는 비용이 비쌉니다. Groq의 하드웨어 인프라와 소프트웨어 플랫폼을 운영하는 데 많은 비용이 소요되며

토큰: prompt=73, completion=320, total=393

--- 모델 비교: llama-3.3-70b-versatile ---
LLM API 수업에서 Groq, OpenAI, Gemini의 비교와 각 모델의 장점 및 한계를 균형 있게 설명하는 것은 중요합니다. Groq는 상대적으로 새로운 모델로서 몇 가지 독특한 특징과 성능을 가지고 있지만,同時으로도 개발 과정과 활용 범위에 일부 제한이 있을 수 있습니다. 아래에 Groq의 장점과 한계를 균형 있게 소개하겠습니다.

### 장점

1. **고성능 컴퓨팅**: Groq는 고성능 컴퓨팅을 지원하며, особенно 대규모 데이터 처리와 복잡한 모델 학습에 강점을 보입니다. 이는 빠른 처리 속도와 효율적인 컴퓨

---
## 8. 실무 레시피


In [10]:
# 레시피 1: 빠른 분류 작업
generate(
    "빠른 분류",
    "다음 문장을 긍정/중립/부정 중 하나로 분류하고 이유를 한 문장으로 말해줘: 수업 속도가 빨라서 조금 어렵지만 재미있다.",
    system="출력은 '분류: ... / 이유: ...' 형식으로 작성하세요.",
    temperature=0.1,
    max_completion_tokens=100,
)



--- 빠른 분류 ---
분류: 긍정 / 이유: 문장에서 '재미있다'라는 긍정적인 표현이 사용되어 긍정적인 감정을 드러내고 있기 때문이다.

토큰: prompt=99, completion=43, total=142


ChatCompletion(id='chatcmpl-5c89d4c3-50f6-456f-bd84-5d9642d055e2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="분류: 긍정 / 이유: 문장에서 '재미있다'라는 긍정적인 표현이 사용되어 긍정적인 감정을 드러내고 있기 때문이다.", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055402, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_d9492c3c54', usage=CompletionUsage(completion_tokens=43, prompt_tokens=99, total_tokens=142, completion_time=0.09391521, completion_tokens_details=None, prompt_time=0.023736487, prompt_tokens_details=None, queue_time=0.177493873, total_time=0.117651697), usage_breakdown=None, x_groq=XGroq(id='req_01kwk64h1sf2ctyjjepw1fmr8h', debug=None, seed=138281478, usage=None))

In [11]:
# 레시피 2: 자연스러운 대화
generate(
    "자연스러운 대화",
    "수업 중 API 오류가 났을 때 학생에게 어떻게 말하면 좋을까?",
    system="따뜻하고 짧은 말투로 답하세요.",
    max_completion_tokens=180,
)



--- 자연스러운 대화 ---
API 오류가 나면 당황하지 말고 상냥하게 대해 주시고 이렇게 말해 주시면 좋을 거야!

"예, 혹시 오류가 났을 수도 있어요. 조금 기다려 주세요. 나는 문제를 찾는 중이에요!"

또는

"오류가 나서 멈췄어요. 잠시 기다려 주세요. 곧 다시 시작할게요!"

토큰: prompt=68, completion=89, total=157


ChatCompletion(id='chatcmpl-06fe21ba-a34d-4817-ac2a-95e63b326a89', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='API 오류가 나면 당황하지 말고 상냥하게 대해 주시고 이렇게 말해 주시면 좋을 거야!\n\n"예, 혹시 오류가 났을 수도 있어요. 조금 기다려 주세요. 나는 문제를 찾는 중이에요!"\n\n또는\n\n"오류가 나서 멈췄어요. 잠시 기다려 주세요. 곧 다시 시작할게요!"', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055402, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_e2c608b1d6', usage=CompletionUsage(completion_tokens=89, prompt_tokens=68, total_tokens=157, completion_time=0.216360681, completion_tokens_details=None, prompt_time=0.003676276, prompt_tokens_details=None, queue_time=0.06823891, total_time=0.220036957), usage_breakdown=None, x_groq=XGroq(id='req_01kwk64hhcf2cv8twtdmak4n7b', debug=None, seed=1672383546, usage=None))

In [12]:
# 레시피 3: 창의적 글쓰기
generate(
    "창의적 글쓰기",
    "LLM API 수업을 홍보하는 짧은 문구 5개를 만들어줘.",
    temperature=1.1,
    top_p=0.95,
    max_completion_tokens=240,
)



--- 창의적 글쓰기 ---
以下는 LLM API 수업을 홍보하는 짧은 문구 5개입니다:

1. "인공지능을 위한 전문 지식과 실무 경험이 있는 강사진과 함께 LLM API 개발을 학습하세요. 함께 성장하여 앞으로의 기술 경쟁력을 높여보세요!"
2. "최신 트렌드에 따라서 개발을 배우는 LLM API 학습과정을 통해 AI 기술의 전반을 학습하세요."
3. "LLM API 개발과정을 통하여 새로운 아이디어와 비즈니스 모델을 구현하고, AI 인공지능의 세계를 탐험해 보세요."
4. "고급 인공지능 기술과 최신 프로그래밍 기술로 LLM API 구현을 학습하세요. 다양한 실전 프로젝트로 실력을 향상해 보세요!"
5. "최신 기술 스택으로 AI 인공지능 개발을 위한 LLM API 프로젝트로 성장하여 비즈니스 아이디어를 구현하고, 창조적인 아이디어를 발견해 보세요!"

토큰: prompt=56, completion=225, total=281


ChatCompletion(id='chatcmpl-8e45bf89-8ad9-4612-bfa5-f087f83866f0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='以下는 LLM API 수업을 홍보하는 짧은 문구 5개입니다:\n\n1. "인공지능을 위한 전문 지식과 실무 경험이 있는 강사진과 함께 LLM API 개발을 학습하세요. 함께 성장하여 앞으로의 기술 경쟁력을 높여보세요!"\n2. "최신 트렌드에 따라서 개발을 배우는 LLM API 학습과정을 통해 AI 기술의 전반을 학습하세요."\n3. "LLM API 개발과정을 통하여 새로운 아이디어와 비즈니스 모델을 구현하고, AI 인공지능의 세계를 탐험해 보세요."\n4. "고급 인공지능 기술과 최신 프로그래밍 기술로 LLM API 구현을 학습하세요. 다양한 실전 프로젝트로 실력을 향상해 보세요!"\n5. "최신 기술 스택으로 AI 인공지능 개발을 위한 LLM API 프로젝트로 성장하여 비즈니스 아이디어를 구현하고, 창조적인 아이디어를 발견해 보세요!"', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055403, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_e2c608b1d6', usage=CompletionUsage(completion_tokens=225, prompt_tokens=56, total_tokens=281, completion_time=0.485585113, completion_tokens_details=Non

In [13]:
# 레시피 4: 코드 생성
generate(
    "코드 생성",
    "python-dotenv로 .env의 GROQ_API_KEY를 읽는 최소 예제를 작성해줘.",
    system="코드는 하나의 블록으로만 작성하고, 코드 아래에 주의점 2개를 붙이세요.",
    max_completion_tokens=340,
)



--- 코드 생성 ---
```python
import os
from dotenv import load_dotenv

# .env 파일을 읽어옵니다.
load_dotenv()

# .env에서 GROQ_API_KEY를 읽습니다.
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# 결과를 출력합니다.
print(GROQ_API_KEY)
```

주의점:
1. `load_dotenv()` 함수는 `.env` 파일을 읽어온 후 환경변수를 읽습니다. 이 함수는 `.env` 파일이 있는 폴더를 현재 폴더로 세팅해줍니다. 만약 `.env` 파일이 다른 폴더에 있다면 이 폴더의 경로를 지정해줘야 합니다.
2. `os.getenv('키')`로 환경변수가 정의되지 않은 경우 `None`을 리턴합니다. 이로 인해 `NoneType` 오류가 발생할 수 있습니다. 따라서 `os.getenv('키')`을 사용하기 전에는 먼저 환경변수가 정의되어있는지만 확인하는 것이 좋습니다.

토큰: prompt=85, completion=220, total=305


ChatCompletion(id='chatcmpl-5dba5834-30fc-40bc-b549-32ac3a711cdd', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="```python\nimport os\nfrom dotenv import load_dotenv\n\n# .env 파일을 읽어옵니다.\nload_dotenv()\n\n# .env에서 GROQ_API_KEY를 읽습니다.\nGROQ_API_KEY = os.getenv('GROQ_API_KEY')\n\n# 결과를 출력합니다.\nprint(GROQ_API_KEY)\n```\n\n주의점:\n1. `load_dotenv()` 함수는 `.env` 파일을 읽어온 후 환경변수를 읽습니다. 이 함수는 `.env` 파일이 있는 폴더를 현재 폴더로 세팅해줍니다. 만약 `.env` 파일이 다른 폴더에 있다면 이 폴더의 경로를 지정해줘야 합니다.\n2. `os.getenv('키')`로 환경변수가 정의되지 않은 경우 `None`을 리턴합니다. 이로 인해 `NoneType` 오류가 발생할 수 있습니다. 따라서 `os.getenv('키')`을 사용하기 전에는 먼저 환경변수가 정의되어있는지만 확인하는 것이 좋습니다.", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1783055404, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_e09ee421cf', usage=CompletionUsage(completion_tokens=220, prompt_token

---
## 9. 조합 비교


In [14]:
comparison_prompt = "Groq API 호출 비용과 지연시간을 줄이는 방법을 5개 알려줘."

settings = [
    {"name": "짧고 안정적", "params": {"max_completion_tokens": 120, "temperature": 0.2}},
    {"name": "균형", "params": {"max_completion_tokens": 220, "temperature": 0.7}},
    {"name": "자세하고 다양", "params": {"max_completion_tokens": 360, "temperature": 1.0, "top_p": 0.95}},
]

for setting in settings:
    generate(setting["name"], comparison_prompt, **setting["params"])



--- 짧고 안정적 ---
Groq API 호출 비용과 지연시간을 줄이는 방법은 다음과 같습니다.

1. **API 호출 최소화**: API 호출을 최소화하는 것은 비용과 지연시간을 줄이는 가장 좋은 방법입니다. API 호출을 최소화하기 위해서는 API를 설계할 때, 필요한 데이터만 가져오도록 설계하는 것이 좋습니다. 또한, 캐싱을 사용하여 이미 가져온 데이터를 재사용하는 것도 좋은 방법입니다.

2. **API 호출 병렬화**: API 호출을 병렬화하는 것은 여러 API를 동

토큰: prompt=57, completion=120, total=177

--- 균형 ---
Groq API 호출 비용과 지연시간을 줄이는 방법에는 다음과 같은 5 가지 방법이 있습니다.

1. **캐싱**: Groq API에 대한 요청을 캐시하는 것은 비용과 지연시간을 줄이는 좋은 방법입니다. 캐싱은 동일한 요청을 여러 번 처리하는 것을 피하고, 자주 요청되는 데이터를 빠르게 제공할 수 있습니다. Groq API에서 캐싱을 사용할 수 있는 방법에는 Memcached, Redis, 또는 In-Memory 캐싱이 있습니다.

2. **병렬 처리**: Groq API의 요청을 병렬로 처리하는 것은 병렬 처리를 사용하는 것에 따라 비용과 지연시간을 줄일 수 있습니다. 여러 요청을 동시에 처리할 수 있기 때문에, 요청을 처리하는 시간이 단축됩니다. Groq API에서 병렬 처리를 사용할 수 있는 방법에는 Worker Pool, 또는 Task Queue가 있습니다.

3. **분산 처리**: Groq API의 요청을 분산 처리하는 것은 여러 서버에 요청을 분산하여

토큰: prompt=57, completion=220, total=277

--- 자세하고 다양 ---
Groq는 인지 플랫폼이기 때문에 API 호출 비용과 지연시간을 줄이는 것은 중요한 문제입니다. 여기서 5개의 방법을 소개합니다.

### 1. 캐싱

캐싱은 가장 일반적인 방법 중 하나입니다. Groq에서는 쿼리 결과를 캐시